<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-07-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [21]:
%pip install mlflow -q

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação baseados em Multi-Layer Perceptrons (MLPs).

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:

- correto
- reproduzível
- rastreável
- criticamente analisado

Além disso, utilizaremos o MLflow para registrar:

- hiperparâmetros
- métricas
- execuções
- comparações
- experimentais

In [22]:
import warnings

warnings.filterwarnings("ignore")

In [23]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [24]:
import time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [25]:
mlflow.set_experiment(
    "assignment"
)

<Experiment: artifact_location='/content/mlruns/1', creation_time=1778955939724, experiment_id='1', last_update_time=1778955939724, lifecycle_stage='active', name='assignment', tags={}, trace_location=None, workspace='default'>

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST` utilizando fetch_openml.
Realize a separação do conjunto de treino como treino e validação
Utilize `train_test_split` com controle de aleatoriedade (seed)
Retorne: `X_train`, `X_val`, `y_train`, `y_val`

Depois responda:
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [26]:
def load_data(seed):
    data = fetch_openml("Fashion-MNIST", version=1, as_frame=False)

    X = data.data.astype(np.float32)
    y = data.target.astype(int)

    # Normalização dos pixels de 0-255 para 0-1
    X = X / 255.0

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )

    return X_train, X_val, y_train, y_val

Sim, é necessário normalizar os dados. O Fashion MNIST possui imagens com pixels variando de 0 a 255. Como o MLP é um modelo baseado em redes neurais, valores muito altos podem dificultar o processo de otimização, tornando o treinamento mais lento ou instável. Ao normalizar os dados para o intervalo entre 0 e 1, o treinamento tende a convergir melhor, com maior estabilidade e melhor desempenho.

# Questão 2

Implemente a função:
`
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
`

## Requisitos:

Utilizar `MLPClassifier` do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [27]:
def train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed,
    max_iter=20,
    batch_size=128
):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        max_iter=max_iter,
        batch_size=batch_size,
        random_state=seed,
        solver="adam",
        shuffle=True
    )

    model.fit(X_train, y_train)

    return model

# Questão 3

Implemente a função:

`evaluate(model, X_test, y_test)`

Ela deve:

- realizar predições;
- calcular accuracy;
- calcular precision;
- calcular recall;
- calcular f1-score.

**Solução**:

In [28]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    }

    return metrics

A função evaluate realiza as predições do modelo no conjunto de validação e calcula as principais métricas de classificação: accuracy, precision, recall e f1-score. Como o problema possui múltiplas classes, foi utilizada a média weighted para considerar a proporção de amostras de cada classe.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow. Devem ser registrados:

Parâmetros
- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

Métricas
- accuracy
- precision
- recall
- f1_score
- training_time

**Solução**:

In [29]:
def run_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    activation,
    hidden_layers,
    learning_rate,
    seed,
    max_iter=20,
    batch_size=128,
    run_name=None
):
    if run_name is None:
        run_name = f"act={activation}_layers={hidden_layers}_lr={learning_rate}"

    with mlflow.start_run(run_name=run_name):
        start_time = time.time()

        model = train_mlp(
            X_train=X_train,
            y_train=y_train,
            activation=activation,
            hidden_layers=hidden_layers,
            learning_rate=learning_rate,
            seed=seed,
            max_iter=max_iter,
            batch_size=batch_size
        )

        training_time = time.time() - start_time

        metrics = evaluate(model, X_val, y_val)

        mlflow.log_param("activation", activation)
        mlflow.log_param("hidden_layers", str(hidden_layers))
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("seed", seed)

        mlflow.log_metric("accuracy", metrics["accuracy"])
        mlflow.log_metric("precision", metrics["precision"])
        mlflow.log_metric("recall", metrics["recall"])
        mlflow.log_metric("f1_score", metrics["f1_score"])
        mlflow.log_metric("training_time", training_time)

        if hasattr(model, "loss_"):
            mlflow.log_metric("final_loss", model.loss_)

        if hasattr(model, "n_iter_"):
            mlflow.log_metric("n_iter", model.n_iter_)

        result = {
            "activation": activation,
            "hidden_layers": hidden_layers,
            "learning_rate": learning_rate,
            "max_iter": max_iter,
            "batch_size": batch_size,
            "training_time": training_time,
            "final_loss": model.loss_ if hasattr(model, "loss_") else None,
            "n_iter": model.n_iter_ if hasattr(model, "n_iter_") else None,
            **metrics
        }

    return model, result

# Questão 5

Compare diferentes funções de ativação.

- logistic
- tanh
- relu

Você deve registrar todos os experimentos utilizando MLflow.

**Solução**:

In [30]:
seed = 42

X_train, X_val, y_train, y_val = load_data(seed)

In [31]:
activations = ["logistic", "tanh", "relu"]

activation_results = []

for activation in activations:
    model, result = run_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation=activation,
        hidden_layers=(64,),
        learning_rate=0.001,
        seed=seed,
        max_iter=20,
        batch_size=128,
        run_name=f"activation_{activation}"
    )

    activation_results.append(result)

activation_results_df = pd.DataFrame(activation_results)
activation_results_df

,activation,hidden_layers,learning_rate,max_iter,batch_size,training_time,final_loss,n_iter,accuracy,precision,recall,f1_score
0,logistic,"(64,)",0.001,20,128,53.752705,0.270075,20,0.882643,0.883139,0.882643,0.882000
1,tanh,"(64,)",0.001,20,128,25.760640,0.224128,20,0.882643,0.884043,0.882643,0.881676
2,relu,"(64,)",0.001,20,128,38.000045,0.241925,20,0.878786,0.880109,0.878786,0.877601


## Responda:
- Qual ativação apresentou melhor convergência?
- Qual ativação apresentou maior estabilidade?
- Houve diferenças significativas de treinamento?

Na comparação entre as funções de ativação, a logistic apresentou o melhor desempenho em termos de f1-score, com resultado ligeiramente superior ao tanh e ao relu.

Em relação à convergência, a tanh apresentou o menor valor de loss final e também o menor tempo de treinamento, indicando um comportamento bastante estável nesse experimento.

A ReLU, apesar de normalmente ser uma função de ativação eficiente para redes neurais, não apresentou o melhor resultado nessa configuração específica com arquitetura (64,) e learning rate 0.001.

Houve diferenças perceptíveis no treinamento, mas não muito grandes entre logistic, tanh e relu em termos de accuracy e f1-score. A principal diferença apareceu no tempo de treinamento e no valor final da loss.

# Questão 6

Compare diferentes arquiteturas de MLP.
`
- (32,)
- (64,)
- (128, 64)
- (256, 128)
`

**Solução**:

In [32]:
architectures = [
    (32,),
    (64,),
    (128, 64),
    (256, 128)
]

architecture_results = []

for hidden_layers in architectures:
    model, result = run_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation="relu",
        hidden_layers=hidden_layers,
        learning_rate=0.001,
        seed=seed,
        max_iter=20,
        batch_size=128,
        run_name=f"architecture_{hidden_layers}"
    )

    architecture_results.append(result)

architecture_results_df = pd.DataFrame(architecture_results)
architecture_results_df

,activation,hidden_layers,learning_rate,max_iter,batch_size,training_time,final_loss,n_iter,accuracy,precision,recall,f1_score
0,relu,"(32,)",0.001,20,128,24.701155,0.289491,20,0.870357,0.874030,0.870357,0.871269
1,relu,"(64,)",0.001,20,128,35.880466,0.241925,20,0.878786,0.880109,0.878786,0.877601
2,relu,"(128, 64)",0.001,20,128,56.758706,0.186544,20,0.889143,0.890762,0.889143,0.887885
3,relu,"(256, 128)",0.001,20,128,103.800493,0.160216,20,0.887071,0.889358,0.887071,0.887236


## Responda:

- Redes maiores sempre melhoraram os resultados?
- Redes maiores sempre melhoraram os resultados?
- Qual arquitetura apresentou melhor tradeoff?

Redes maiores não necessariamente melhoraram os resultados. A arquitetura (32,) apresentou o menor desempenho, enquanto a arquitetura (128, 64) obteve o melhor f1-score. A arquitetura (256, 128), mesmo sendo maior, não superou a (128, 64) e apresentou maior tempo de treinamento.

Assim, a arquitetura (128, 64) apresentou o melhor tradeoff, pois teve o melhor desempenho geral sem o custo computacional mais alto da rede (256, 128).

# Questão 7

Analise o impacto do learning rate.
- 0.1
- 0.01
- 0.001

In [33]:
learning_rates = [0.1, 0.01, 0.001]

learning_rate_results = []

for lr in learning_rates:
    model, result = run_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation="relu",
        hidden_layers=(128, 64),
        learning_rate=lr,
        seed=seed,
        max_iter=20,
        batch_size=128,
        run_name=f"learning_rate_{lr}"
    )

    learning_rate_results.append(result)

learning_rate_results_df = pd.DataFrame(learning_rate_results)
learning_rate_results_df

,activation,hidden_layers,learning_rate,max_iter,batch_size,training_time,final_loss,n_iter,accuracy,precision,recall,f1_score
0,relu,"(128, 64)",0.100,20,128,86.009055,1.821901,13,0.199857,0.148389,0.199857,0.075901
1,relu,"(128, 64)",0.010,20,128,136.737805,0.284554,20,0.869286,0.873572,0.869286,0.866746
2,relu,"(128, 64)",0.001,20,128,66.289252,0.186544,20,0.889143,0.890762,0.889143,0.887885


## Responda:
- O treinamento ficou instável?
- Houve dificuldade de convergência?
- Qual learning rate apresentou melhor comportamento?

O treinamento ficou instável com learning rate 0.1, pois o desempenho caiu significativamente e o f1-score ficou muito baixo.

Com learning rate 0.01, o modelo conseguiu aprender, mas teve desempenho inferior ao learning rate 0.001.

O learning rate 0.001 apresentou o melhor comportamento, com maior accuracy, maior f1-score e menor loss final entre os valores testados.

# Questão 8

- Qual ativação apresentou melhor desempenho?
- Qual arquitetura apresentou melhor tradeoff?
- Qual learning rate apresentou maior estabilidade?
- Houve overfitting?
- Qual configuração apresentou melhor resultado final?
- Quais foram as principais dificuldades observadas?


In [34]:
all_results_df = pd.concat(
    [
        activation_results_df,
        architecture_results_df,
        learning_rate_results_df
    ],
    ignore_index=True
)

all_results_df.sort_values(by="f1_score", ascending=False)

,activation,hidden_layers,learning_rate,max_iter,batch_size,training_time,final_loss,n_iter,accuracy,precision,recall,f1_score
9,relu,"(128, 64)",0.001,20,128,66.289252,0.186544,20,0.889143,0.890762,0.889143,0.887885
5,relu,"(128, 64)",0.001,20,128,56.758706,0.186544,20,0.889143,0.890762,0.889143,0.887885
6,relu,"(256, 128)",0.001,20,128,103.800493,0.160216,20,0.887071,0.889358,0.887071,0.887236
0,logistic,"(64,)",0.001,20,128,53.752705,0.270075,20,0.882643,0.883139,0.882643,0.882000
1,tanh,"(64,)",0.001,20,128,25.760640,0.224128,20,0.882643,0.884043,0.882643,0.881676
2,relu,"(64,)",0.001,20,128,38.000045,0.241925,20,0.878786,0.880109,0.878786,0.877601
4,relu,"(64,)",0.001,20,128,35.880466,0.241925,20,0.878786,0.880109,0.878786,0.877601
3,relu,"(32,)",0.001,20,128,24.701155,0.289491,20,0.870357,0.874030,0.870357,0.871269
8,relu,"(128, 64)",0.010,20,128,136.737805,0.284554,20,0.869286,0.873572,0.869286,0.866746
7,relu,"(128, 64)",0.100,20,128,86.009055,1.821901,13,0.199857,0.148389,0.199857,0.075901


Na comparação isolada entre funções de ativação usando a arquitetura (64,) e learning rate 0.001, a ativação logistic apresentou o melhor f1-score. Porém, considerando todos os experimentos realizados, a melhor configuração final utilizou ReLU com arquitetura (128, 64) e learning rate 0.001.

A arquitetura com melhor tradeoff foi a (128, 64), pois apresentou o melhor f1-score sem ter o maior tempo de treinamento. A arquitetura (256, 128) teve desempenho muito próximo, mas exigiu mais tempo e não trouxe ganho significativo.

O learning rate mais estável foi 0.001, pois apresentou o melhor desempenho e menor loss final. O learning rate 0.1 se mostrou instável, com queda muito grande nas métricas.

Não há evidência forte de overfitting apenas com os resultados disponíveis, pois a análise foi feita principalmente com métricas de validação. Para confirmar overfitting com mais segurança, seria necessário comparar o desempenho no treino e na validação.

A melhor configuração final foi ReLU com arquitetura (128, 64) e learning rate 0.001, pois obteve o melhor f1-score geral.

As principais dificuldades observadas foram o tempo de treinamento dos modelos maiores, a sensibilidade do MLP ao learning rate e a necessidade de normalizar os dados para melhorar a estabilidade da convergência.

In [35]:
all_results_df[
    [
        "activation",
        "hidden_layers",
        "learning_rate",
        "accuracy",
        "precision",
        "recall",
        "f1_score",
        "training_time",
        "final_loss",
        "n_iter"
    ]
].sort_values(by="f1_score", ascending=False)

,activation,hidden_layers,learning_rate,accuracy,precision,recall,f1_score,training_time,final_loss,n_iter
9,relu,"(128, 64)",0.001,0.889143,0.890762,0.889143,0.887885,66.289252,0.186544,20
5,relu,"(128, 64)",0.001,0.889143,0.890762,0.889143,0.887885,56.758706,0.186544,20
6,relu,"(256, 128)",0.001,0.887071,0.889358,0.887071,0.887236,103.800493,0.160216,20
0,logistic,"(64,)",0.001,0.882643,0.883139,0.882643,0.882000,53.752705,0.270075,20
1,tanh,"(64,)",0.001,0.882643,0.884043,0.882643,0.881676,25.760640,0.224128,20
2,relu,"(64,)",0.001,0.878786,0.880109,0.878786,0.877601,38.000045,0.241925,20
4,relu,"(64,)",0.001,0.878786,0.880109,0.878786,0.877601,35.880466,0.241925,20
3,relu,"(32,)",0.001,0.870357,0.874030,0.870357,0.871269,24.701155,0.289491,20
8,relu,"(128, 64)",0.010,0.869286,0.873572,0.869286,0.866746,136.737805,0.284554,20
7,relu,"(128, 64)",0.100,0.199857,0.148389,0.199857,0.075901,86.009055,1.821901,13
